<a href="https://colab.research.google.com/github/DarioCorona/personal/blob/root/Evaluacion_Modulo5_Machine_Learning_Mantenimiento_Colab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación integradora - Módulo 5

## Machine Learning aplicado al mantenimiento predictivo

**Ingenium - Ciencia de Datos aplicada al Mantenimiento Predictivo**

Esta evaluación integra formulación supervisada, clasificación, regresión, evaluación, optimización, interpretación y persistencia de modelos. Trabajarás con el conjunto real **Condition Monitoring of Hydraulic Systems**, publicado por UCI.

> Completa únicamente las celdas marcadas con **RESPUESTA DEL ESTUDIANTE**. No modifiques la preparación de datos ni el calificador.

## Reglas de trabajo

- Ejecuta las celdas en orden.
- Conserva los nombres de variables solicitados.
- No agregues la variable objetivo, sus copias ni etiquetas del perfil dentro de `X`.
- Utiliza `random_state=42` cuando se indique.
- La prueba no debe participar en la selección de hiperparámetros.
- Las métricas deben calcularse sobre los conjuntos de prueba.
- Guarda el notebook con las salidas y la tabla final de calificación visibles.
- El puntaje máximo es **20 puntos**.

La calificación automática comprueba estructura y resultados. El profesor puede revisar además claridad, coherencia técnica e integridad académica.

## 0. Datos del estudiante

Completa la siguiente celda antes de comenzar.

In [14]:
# RESPUESTA DEL ESTUDIANTE
NOMBRE_COMPLETO = "DARIO ISIDRO CORONA SILVA"
GRUPO = "Programa de Certificación como Analista de Mantenimiento Predictivo - STD MAN PC ANAMANP 2026 II MEX"
CORREO = "dicssug@gmail.com"


## 1. Preparación del entorno y datos

La celda siguiente descarga automáticamente el ZIP oficial de UCI. El archivo pesa aproximadamente 73 MB. Cada fila final representa un ciclo de trabajo del banco hidráulico y cada característica es el promedio de un sensor físico durante ese ciclo.

**No modifiques esta celda.**

In [15]:
# CELDA DE PREPARACIÓN - NO MODIFICAR
from pathlib import Path
from zipfile import ZipFile
import gc
import os
import urllib.request

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
DATA_URLS = [
    "https://zenodo.org/records/1323611/files/data.zip?download=1",
    "https://archive.ics.uci.edu/static/public/447/condition%2Bmonitoring%2Bof%2Bhydraulic%2Bsystems.zip",
]
ZIP_PATH = Path("condition_monitoring_hydraulic_systems.zip")

def zip_valido(ruta):
    if not ruta.exists():
        return False
    try:
        with ZipFile(ruta) as archivo:
            return any(Path(nombre).name.lower() == "profile.txt" for nombre in archivo.namelist())
    except Exception:
        return False

if not zip_valido(ZIP_PATH):
    ZIP_PATH.unlink(missing_ok=True)
    errores_descarga = []
    for url in DATA_URLS:
        temporal = ZIP_PATH.with_suffix(".partial")
        temporal.unlink(missing_ok=True)
        try:
            print("Descargando el conjunto hidráulico real...")
            urllib.request.urlretrieve(url, temporal)
            if not zip_valido(temporal):
                raise RuntimeError("La descarga no produjo un ZIP válido.")
            temporal.replace(ZIP_PATH)
            break
        except Exception as error:
            errores_descarga.append(f"{url}: {error}")
            temporal.unlink(missing_ok=True)
    else:
        raise RuntimeError("No fue posible descargar el conjunto de datos. " + " | ".join(errores_descarga))

SENSORES_FISICOS = [
    "PS1", "PS2", "PS3", "PS4", "PS5", "PS6",
    "EPS1", "FS1", "FS2", "TS1", "TS2", "TS3", "TS4", "VS1",
]

def buscar_miembro(zip_file, nombre_archivo):
    coincidencias = [
        nombre for nombre in zip_file.namelist()
        if Path(nombre).name.lower() == nombre_archivo.lower()
    ]
    if not coincidencias:
        raise FileNotFoundError(f"No se encontró {nombre_archivo} dentro del ZIP.")
    return coincidencias[0]

def leer_matriz(zip_file, nombre_archivo):
    miembro = buscar_miembro(zip_file, nombre_archivo)
    return pd.read_csv(
        zip_file.open(miembro),
        sep=r"\s+",
        header=None,
        dtype=np.float32,
    )

with ZipFile(ZIP_PATH) as archivo_zip:
    perfil = leer_matriz(archivo_zip, "profile.txt")
    perfil.columns = [
        "cooler_condition_pct",
        "valve_condition_pct",
        "pump_leakage",
        "accumulator_bar",
        "stable_flag",
    ]

    resumen_sensores = {}
    for sensor in SENSORES_FISICOS:
        matriz = leer_matriz(archivo_zip, f"{sensor}.txt")
        resumen_sensores[f"{sensor}_mean"] = matriz.mean(axis=1).astype("float32")
        del matriz
        gc.collect()
        print(f"Preparado: {sensor}")

    # CE es una señal virtual continua; se usa exclusivamente como objetivo de regresión.
    ce = leer_matriz(archivo_zip, "CE.txt")
    cooling_efficiency_pct = ce.mean(axis=1).astype("float32")
    del ce
    gc.collect()

datos = pd.DataFrame(resumen_sensores)
datos.insert(0, "cycle_id", np.arange(1, len(datos) + 1))
datos = pd.concat([datos, perfil.reset_index(drop=True)], axis=1)
datos["cooling_efficiency_pct"] = cooling_efficiency_pct.reset_index(drop=True)

FEATURE_COLUMNS = [f"{sensor}_mean" for sensor in SENSORES_FISICOS]
TARGET_CLASSIFICATION = "pump_leakage"
TARGET_REGRESSION = "cooling_efficiency_pct"
PROFILE_COLUMNS = [
    "cooler_condition_pct", "valve_condition_pct", "pump_leakage",
    "accumulator_bar", "stable_flag",
]

print()
print("Preparación terminada")
print("Dimensiones:", datos.shape)
print("Características autorizadas:", len(FEATURE_COLUMNS))
print("Clases de fuga interna:", sorted(datos[TARGET_CLASSIFICATION].unique().tolist()))
display(datos.head())


Preparado: PS1
Preparado: PS2
Preparado: PS3
Preparado: PS4
Preparado: PS5
Preparado: PS6
Preparado: EPS1
Preparado: FS1
Preparado: FS2
Preparado: TS1
Preparado: TS2
Preparado: TS3
Preparado: TS4
Preparado: VS1

Preparación terminada
Dimensiones: (2205, 21)
Características autorizadas: 14
Clases de fuga interna: [0.0, 1.0, 2.0]


,cycle_id,PS1_mean,PS2_mean,PS3_mean,PS4_mean,PS5_mean,PS6_mean,EPS1_mean,FS1_mean,FS2_mean,...,TS2_mean,TS3_mean,TS4_mean,VS1_mean,cooler_condition_pct,valve_condition_pct,pump_leakage,accumulator_bar,stable_flag,cooling_efficiency_pct
0,1,160.673752,109.466820,1.991467,0.0,9.842184,9.728086,2538.977783,6.709809,10.304590,...,40.978760,38.471008,31.745258,0.576950,3.0,100.0,0.0,130.0,1.0,39.601345
1,2,160.602661,109.354851,1.976229,0.0,9.635093,9.529501,2531.571777,6.715314,10.403094,...,41.532764,38.978962,34.493858,0.565850,3.0,100.0,0.0,130.0,1.0,25.786434
2,3,160.347488,109.158760,1.972215,0.0,9.530541,9.427933,2519.982666,6.718522,10.366256,...,42.442444,39.631954,35.646160,0.576533,3.0,100.0,0.0,130.0,1.0,22.218231
3,4,160.188141,109.064812,1.946568,0.0,9.438860,9.337433,2511.614258,6.720563,10.302678,...,43.403984,40.403393,36.579468,0.569267,3.0,100.0,0.0,130.0,1.0,20.459816
4,5,159.999527,108.931580,1.922695,0.0,9.358758,9.260631,2503.528564,6.690309,10.237757,...,44.332756,41.310555,37.427906,0.577367,3.0,100.0,0.0,130.0,1.0,19.787018


### Diagnóstico de control

Ejecuta la siguiente celda y confirma que existen 2,205 ciclos, 14 características autorizadas y tres clases para `pump_leakage`.

In [16]:
# CONTROL - NO MODIFICAR
control_datos = pd.DataFrame({
    "indicador": [
        "ciclos", "caracteristicas", "faltantes_en_X",
        "clases_bomba", "objetivo_regresion_min", "objetivo_regresion_max",
    ],
    "valor": [
        len(datos),
        len(FEATURE_COLUMNS),
        int(datos[FEATURE_COLUMNS].isna().sum().sum()),
        int(datos[TARGET_CLASSIFICATION].nunique()),
        float(datos[TARGET_REGRESSION].min()),
        float(datos[TARGET_REGRESSION].max()),
    ],
})
display(control_datos)
display(datos[TARGET_CLASSIFICATION].value_counts().sort_index().rename("ciclos"))


,indicador,valor
0,ciclos,2205.000000
1,caracteristicas,14.000000
2,faltantes_en_X,0.000000
3,clases_bomba,3.000000
4,objetivo_regresion_min,17.555983
5,objetivo_regresion_max,47.903664


,ciclos
pump_leakage,
0.0,1221
1.0,492
2.0,492


## 2. Formulación de los problemas - 2 puntos

Construye exactamente:

- `X`: solamente las columnas incluidas en `FEATURE_COLUMNS`.
- `y_clasificacion`: la columna `pump_leakage`.
- `y_regresion`: la columna continua `cooling_efficiency_pct`.
- `revision_fuga`: diccionario con las claves `columnas_objetivo_en_X` y `hay_fuga`.

`columnas_objetivo_en_X` debe listar cualquier objetivo o etiqueta del perfil incluida por error en `X`. `hay_fuga` debe ser un booleano.

In [17]:
# RESPUESTA DEL ESTUDIANTE - TAREA 1
# Escribe tu solución debajo.




In [18]:
X = datos[FEATURE_COLUMNS]
y_clasificacion = datos[TARGET_CLASSIFICATION]
y_regresion = datos[TARGET_REGRESSION]

# Verificar si alguna columna objetivo o de perfil está en FEATURE_COLUMNS
columnas_objetivo_en_X = []
for col in PROFILE_COLUMNS + [TARGET_REGRESSION]:
    if col in FEATURE_COLUMNS:
        columnas_objetivo_en_X.append(col)

hay_fuga = bool(columnas_objetivo_en_X)

revision_fuga = {
    "columnas_objetivo_en_X": columnas_objetivo_en_X,
    "hay_fuga": hay_fuga,
}

# Mostrar las primeras filas de X y los objetivos para verificación
print("Primeras 5 filas de X:")
display(X.head())
print("Primeras 5 filas de y_clasificacion:")
display(y_clasificacion.head())
print("Primeras 5 filas de y_regresion:")
display(y_regresion.head())
print("Revisión de fuga:")
display(revision_fuga)

Primeras 5 filas de X:


,PS1_mean,PS2_mean,PS3_mean,PS4_mean,PS5_mean,PS6_mean,EPS1_mean,FS1_mean,FS2_mean,TS1_mean,TS2_mean,TS3_mean,TS4_mean,VS1_mean
0,160.673752,109.466820,1.991467,0.0,9.842184,9.728086,2538.977783,6.709809,10.304590,35.621986,40.978760,38.471008,31.745258,0.576950
1,160.602661,109.354851,1.976229,0.0,9.635093,9.529501,2531.571777,6.715314,10.403094,36.676975,41.532764,38.978962,34.493858,0.565850
2,160.347488,109.158760,1.972215,0.0,9.530541,9.427933,2519.982666,6.718522,10.366256,37.880802,42.442444,39.631954,35.646160,0.576533
3,160.188141,109.064812,1.946568,0.0,9.438860,9.337433,2511.614258,6.720563,10.302678,38.879044,43.403984,40.403393,36.579468,0.569267
4,159.999527,108.931580,1.922695,0.0,9.358758,9.260631,2503.528564,6.690309,10.237757,39.803928,44.332756,41.310555,37.427906,0.577367


Primeras 5 filas de y_clasificacion:


,pump_leakage
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0


Primeras 5 filas de y_regresion:


,cooling_efficiency_pct
0,39.601345
1,25.786434
2,22.218231
3,20.459816
4,19.787018


Revisión de fuga:


{'columnas_objetivo_en_X': [], 'hay_fuga': False}

## 3. Particiones de entrenamiento y prueba - 2 puntos

### Clasificación

Utiliza `train_test_split` con 25% para prueba, `random_state=42` y estratificación por clase. Conserva estos nombres:

- `X_train_cls`, `X_test_cls`, `y_train_cls`, `y_test_cls`

### Regresión

Reserva cronológicamente el último 25% de los ciclos como prueba, sin barajar. Conserva:

- `X_train_reg`, `X_test_reg`, `y_train_reg`, `y_test_reg`

No ajustes transformaciones antes de realizar las particiones.

In [19]:
# RESPUESTA DEL ESTUDIANTE - TAREA 2
# Escribe tu solución debajo.



In [20]:
# Partición para clasificación
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X,
    y_clasificacion,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_clasificacion
)

# Partición para regresión (cronológica)
# Calcular el índice de división para el 25% final
split_index_reg = int(len(X) * 0.75)

X_train_reg = X.iloc[:split_index_reg]
X_test_reg = X.iloc[split_index_reg:]
y_train_reg = y_regresion.iloc[:split_index_reg]
y_test_reg = y_regresion.iloc[split_index_reg:]

print("Dimensiones de los conjuntos de clasificación:")
print(f"X_train_cls: {X_train_cls.shape}")
print(f"X_test_cls: {X_test_cls.shape}")
print(f"y_train_cls: {y_train_cls.shape}")
print(f"y_test_cls: {y_test_cls.shape}")

print("\nDimensiones de los conjuntos de regresión:")
print(f"X_train_reg: {X_train_reg.shape}")
print(f"X_test_reg: {X_test_reg.shape}")
print(f"y_train_reg: {y_train_reg.shape}")
print(f"y_test_reg: {y_test_reg.shape}")

Dimensiones de los conjuntos de clasificación:
X_train_cls: (1653, 14)
X_test_cls: (552, 14)
y_train_cls: (1653,)
y_test_cls: (552,)

Dimensiones de los conjuntos de regresión:
X_train_reg: (1653, 14)
X_test_reg: (552, 14)
y_train_reg: (1653,)
y_test_reg: (552,)


## 4. Clasificación de la fuga interna de la bomba - 5 puntos

1. Entrena `modelo_base_cls` con `DummyClassifier(strategy="most_frequent")`.
2. Crea y entrena tres modelos dentro del diccionario `modelos_clasificacion`:
   - regresión logística con escalamiento dentro de un `Pipeline`;
   - árbol de decisión;
   - Random Forest.
3. Guarda las predicciones en el diccionario `predicciones_clasificacion` usando las mismas claves.
4. Construye `resultados_clasificacion` con las columnas:
   - `modelo`, `accuracy`, `precision_macro`, `recall_macro`, `f1_macro`.
   - Incluye el modelo base y los tres modelos entrenados.
5. Selecciona un modelo en `modelo_clasificacion_seleccionado`.
6. Calcula `matriz_confusion` para el modelo seleccionado.

Todas las métricas deben calcularse sobre `y_test_cls`. Usa `zero_division=0` cuando corresponda.

In [21]:
# RESPUESTA DEL ESTUDIANTE - TAREA 3
# Escribe tu solución debajo.



In [22]:
# 1. Entrena modelo_base_cls con DummyClassifier(strategy="most_frequent")
modelo_base_cls = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
modelo_base_cls.fit(X_train_cls, y_train_cls)

# 2. Crea y entrena tres modelos dentro del diccionario modelos_clasificacion
modelos_clasificacion = {}

# Regresión Logística con escalado en un Pipeline
modelo_log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=RANDOM_STATE, solver='liblinear'))
])
modelo_log_reg.fit(X_train_cls, y_train_cls)
modelos_clasificacion['Regresión Logística'] = modelo_log_reg

# Árbol de Decisión
modelo_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
modelo_tree.fit(X_train_cls, y_train_cls)
modelos_clasificacion['Árbol de Decisión'] = modelo_tree

# Random Forest
modelo_rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
modelo_rf.fit(X_train_cls, y_train_cls)
modelos_clasificacion['Random Forest'] = modelo_rf

# 3. Guarda las predicciones en el diccionario predicciones_clasificacion
predicciones_clasificacion = {}
predicciones_clasificacion['Dummy'] = modelo_base_cls.predict(X_test_cls)
for nombre, modelo in modelos_clasificacion.items():
    predicciones_clasificacion[nombre] = modelo.predict(X_test_cls)

# 4. Construye resultados_clasificacion con las columnas especificadas
resultados_clasificacion = pd.DataFrame(columns=[
    'modelo', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro'
])

def calcular_metricas_cls(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }

# Métricas para el modelo base
metricas_base = calcular_metricas_cls(y_test_cls, predicciones_clasificacion['Dummy'])
resultados_clasificacion.loc[0] = ['Dummy Classifier', metricas_base['accuracy'],
                                   metricas_base['precision_macro'], metricas_base['recall_macro'],
                                   metricas_base['f1_macro']]

# Métricas para los modelos entrenados
for i, (nombre, preds) in enumerate(predicciones_clasificacion.items()):
    if nombre == 'Dummy':
        continue # Ya agregado
    metricas = calcular_metricas_cls(y_test_cls, preds)
    resultados_clasificacion.loc[i+1] = [nombre, metricas['accuracy'],
                                        metricas['precision_macro'], metricas['recall_macro'],
                                        metricas['f1_macro']]

display(resultados_clasificacion)

# 5. Selecciona un modelo en modelo_clasificacion_seleccionado
# Basándonos en f1_macro (un buen balance entre precisión y recall para clases desbalanceadas)
mejor_f1 = resultados_clasificacion['f1_macro'].max()
mejor_modelo_nombre = resultados_clasificacion.loc[resultados_clasificacion['f1_macro'] == mejor_f1, 'modelo'].iloc[0]

# Si el mejor modelo es el Dummy Classifier, seleccionar el siguiente mejor modelo no dummy
if mejor_modelo_nombre == 'Dummy Classifier' and len(resultados_clasificacion) > 1:
    # Obtener el segundo mejor f1_macro de los modelos no dummy
    best_f1_non_dummy = resultados_clasificacion[resultados_clasificacion['modelo'] != 'Dummy Classifier']['f1_macro'].max()
    mejor_modelo_nombre = resultados_clasificacion.loc[(resultados_clasificacion['f1_macro'] == best_f1_non_dummy) & (resultados_clasificacion['modelo'] != 'Dummy Classifier'), 'modelo'].iloc[0]

# Obtener el objeto del modelo seleccionado
if mejor_modelo_nombre == 'Dummy Classifier':
    modelo_clasificacion_seleccionado = modelo_base_cls
else:
    # Necesitamos una forma de mapear el nombre del DataFrame a la clave en modelos_clasificacion
    # Asumiendo que los nombres en el DataFrame coinciden con las claves del diccionario
    modelo_clasificacion_seleccionado = modelos_clasificacion[mejor_modelo_nombre]

print(f"\nModelo de clasificación seleccionado: {mejor_modelo_nombre}")

# 6. Calcula matriz_confusion para el modelo seleccionado
predicciones_seleccionadas = modelo_clasificacion_seleccionado.predict(X_test_cls)
matriz_confusion = confusion_matrix(y_test_cls, predicciones_seleccionadas)

print("\nMatriz de Confusión para el modelo seleccionado:")
display(pd.DataFrame(matriz_confusion, index=[f'Real {c}' for c in sorted(y_test_cls.unique())], columns=[f'Pred {c}' for c in sorted(y_test_cls.unique())]))

,modelo,accuracy,precision_macro,recall_macro,f1_macro
0,Dummy Classifier,0.554348,0.184783,0.333333,0.237762
2,Regresión Logística,0.934783,0.929130,0.904060,0.908256
3,Árbol de Decisión,0.987319,0.982587,0.981030,0.981802
4,Random Forest,0.992754,0.989268,0.990781,0.989996



Modelo de clasificación seleccionado: Random Forest

Matriz de Confusión para el modelo seleccionado:


,Pred 0.0,Pred 1.0,Pred 2.0
Real 0.0,305,0,1
Real 1.0,0,121,2
Real 2.0,0,1,122


## 5. Regresión de la eficiencia de enfriamiento - 4 puntos

1. Entrena `modelo_base_reg` con `DummyRegressor(strategy="mean")`.
2. Crea y entrena tres modelos en `modelos_regresion`:
   - regresión lineal;
   - árbol de regresión;
   - Random Forest de regresión.
3. Guarda las predicciones en `predicciones_regresion`.
4. Construye `resultados_regresion` con:
   - `modelo`, `MAE`, `RMSE`, `R2`.
   - Incluye el modelo base y los tres modelos.
5. Guarda el modelo elegido en `modelo_regresion_seleccionado`.

Calcula las métricas sobre `y_test_reg`. RMSE debe conservar las mismas unidades del objetivo.

In [23]:
# RESPUESTA DEL ESTUDIANTE - TAREA 4
# Escribe tu solución debajo.



In [24]:
# 1. Entrena modelo_base_reg con DummyRegressor(strategy="mean")
modelo_base_reg = DummyRegressor(strategy="mean")
modelo_base_reg.fit(X_train_reg, y_train_reg)

# 2. Crea y entrena tres modelos dentro del diccionario modelos_regresion
modelos_regresion = {}

# Regresión Lineal
modelo_lin_reg = LinearRegression()
modelo_lin_reg.fit(X_train_reg, y_train_reg)
modelos_regresion['Regresión Lineal'] = modelo_lin_reg

# Árbol de Regresión
modelo_tree_reg = DecisionTreeRegressor(random_state=RANDOM_STATE)
modelo_tree_reg.fit(X_train_reg, y_train_reg)
modelos_regresion['Árbol de Regresión'] = modelo_tree_reg

# Random Forest de Regresión
modelo_rf_reg = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
modelo_rf_reg.fit(X_train_reg, y_train_reg)
modelos_regresion['Random Forest Regresión'] = modelo_rf_reg

# 3. Guarda las predicciones en el diccionario predicciones_regresion
predicciones_regresion = {}
predicciones_regresion['Dummy'] = modelo_base_reg.predict(X_test_reg)
for nombre, modelo in modelos_regresion.items():
    predicciones_regresion[nombre] = modelo.predict(X_test_reg)

# 4. Construye resultados_regresion con las columnas especificadas
resultados_regresion = pd.DataFrame(columns=['modelo', 'MAE', 'RMSE', 'R2'])

def calcular_metricas_reg(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

# Métricas para el modelo base
metricas_base_reg = calcular_metricas_reg(y_test_reg, predicciones_regresion['Dummy'])
resultados_regresion.loc[0] = ['Dummy Regressor', metricas_base_reg['MAE'],
                                   metricas_base_reg['RMSE'], metricas_base_reg['R2']]

# Métricas para los modelos entrenados
for i, (nombre, preds) in enumerate(predicciones_regresion.items()):
    if nombre == 'Dummy':
        continue # Ya agregado
    metricas_reg = calcular_metricas_reg(y_test_reg, preds)
    resultados_regresion.loc[i+1] = [nombre, metricas_reg['MAE'],
                                        metricas_reg['RMSE'], metricas_reg['R2']]

display(resultados_regresion)

# 5. Guarda el modelo elegido en modelo_regresion_seleccionado
# Seleccionar el modelo con el menor RMSE
mejor_rmse = resultados_regresion['RMSE'].min()
mejor_modelo_reg_nombre = resultados_regresion.loc[resultados_regresion['RMSE'] == mejor_rmse, 'modelo'].iloc[0]

# Si el mejor modelo es el Dummy Regressor, seleccionar el siguiente mejor modelo no dummy
if mejor_modelo_reg_nombre == 'Dummy Regressor' and len(resultados_regresion) > 1:
    # Obtener el segundo mejor RMSE de los modelos no dummy
    best_rmse_non_dummy = resultados_regresion[resultados_regresion['modelo'] != 'Dummy Regressor']['RMSE'].min()
    mejor_modelo_reg_nombre = resultados_regresion.loc[(resultados_regresion['RMSE'] == best_rmse_non_dummy) & (resultados_regresion['modelo'] != 'Dummy Regressor'), 'modelo'].iloc[0]

# Obtener el objeto del modelo seleccionado
if mejor_modelo_reg_nombre == 'Dummy Regressor':
    modelo_regresion_seleccionado = modelo_base_reg
else:
    # Mapear el nombre del DataFrame a la clave en modelos_regresion
    modelo_regresion_seleccionado = modelos_regresion[mejor_modelo_reg_nombre]

print(f"\nModelo de regresión seleccionado: {mejor_modelo_reg_nombre}")

,modelo,MAE,RMSE,R2
0,Dummy Regressor,20.984241,20.987292,-3438.292725
2,Regresión Lineal,1.224481,1.366219,-13.574613
3,Árbol de Regresión,0.649956,1.474084,-15.966835
4,Random Forest Regresión,0.487909,0.789269,-3.864149



Modelo de regresión seleccionado: Random Forest Regresión


## 6. Optimización sin utilizar la prueba - 4 puntos

Optimiza un Random Forest para clasificación.

1. Crea `pipeline_busqueda` con imputación por mediana y `RandomForestClassifier(random_state=42, n_jobs=-1)`.
2. Define `param_grid` con al menos dos valores para `n_estimators`, `max_depth` y `min_samples_leaf`.
3. Crea `cv_estratificada` con `StratifiedKFold`, al menos 3 folds, barajado y `random_state=42`.
4. Crea y ajusta `busqueda` mediante `GridSearchCV` usando `scoring="f1_macro"` y solamente el conjunto de entrenamiento.
5. Guarda `modelo_optimizado = busqueda.best_estimator_`.
6. Calcula sobre la prueba `metricas_optimizadas`, un diccionario con `accuracy`, `precision_macro`, `recall_macro` y `f1_macro`.

La prueba se utiliza una sola vez después de terminar la búsqueda.

In [25]:
# RESPUESTA DEL ESTUDIANTE - TAREA 5
# Escribe tu solución debajo.



In [26]:
# 1. Crea pipeline_busqueda
pipeline_busqueda = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

# 2. Define param_grid
param_grid = {
    'model__n_estimators': [50, 100, 200],  # Al menos dos valores
    'model__max_depth': [10, 20, None],     # Al menos dos valores
    'model__min_samples_leaf': [1, 5, 10]   # Al menos dos valores
}

# 3. Crea cv_estratificada
cv_estratificada = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# 4. Crea y ajusta busqueda mediante GridSearchCV
busqueda = GridSearchCV(
    pipeline_busqueda,
    param_grid,
    scoring='f1_macro',
    cv=cv_estratificada,
    n_jobs=-1, # Utiliza todos los núcleos disponibles
    verbose=1 # Para ver el progreso
)

# Ajustar solo en el conjunto de entrenamiento
busqueda.fit(X_train_cls, y_train_cls)

# 5. Guarda modelo_optimizado
modelo_optimizado = busqueda.best_estimator_

print(f"Mejores parámetros encontrados: {busqueda.best_params_}")
print(f"Mejor puntuación f1_macro en CV: {busqueda.best_score_:.4f}")

# 6. Calcula metricas_optimizadas sobre la prueba
y_pred_optimizadas = modelo_optimizado.predict(X_test_cls)

metricas_optimizadas = {
    'accuracy': accuracy_score(y_test_cls, y_pred_optimizadas),
    'precision_macro': precision_score(y_test_cls, y_pred_optimizadas, average='macro', zero_division=0),
    'recall_macro': recall_score(y_test_cls, y_pred_optimizadas, average='macro', zero_division=0),
    'f1_macro': f1_score(y_test_cls, y_pred_optimizadas, average='macro', zero_division=0)
}

print("\nMétricas del modelo optimizado en el conjunto de prueba:")
display(pd.DataFrame([metricas_optimizadas]))

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Mejores parámetros encontrados: {'model__max_depth': 20, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Mejor puntuación f1_macro en CV: 0.9884

Métricas del modelo optimizado en el conjunto de prueba:


,accuracy,precision_macro,recall_macro,f1_macro
0,0.992754,0.989268,0.990781,0.989996


## 7. Interpretación y persistencia - 2 puntos

1. Obtén la importancia del Random Forest optimizado.
2. Construye `importancias`, un DataFrame ordenado de mayor a menor con las columnas `caracteristica` e `importancia`. Conserva al menos las diez primeras.
3. Guarda el pipeline completo en la ruta `RUTA_MODELO = "modelo_modulo5.joblib"` mediante `joblib.dump`.
4. Comprueba que el archivo puede cargarse y producir predicciones.

La importancia refleja cómo utiliza variables el modelo; no demuestra causalidad física.

In [27]:
# RESPUESTA DEL ESTUDIANTE - TAREA 6
# Escribe tu solución debajo.



In [28]:
# 1. Obtén la importancia del Random Forest optimizado.
# El modelo optimizado es un pipeline, el Random Forest está en el paso 'model'
importances_raw = modelo_optimizado.named_steps['model'].feature_importances_

# 2. Construye importancias, un DataFrame ordenado de mayor a menor
importancias = pd.DataFrame({
    'caracteristica': X_train_cls.columns,
    'importancia': importances_raw
}).sort_values(by='importancia', ascending=False)

# Conservar al menos las diez primeras (en este caso, se muestran todas si hay menos de 10, o las 10 primeras)
display(importancias.head(10))

# 3. Guarda el pipeline completo en la ruta RUTA_MODELO = "modelo_modulo5.joblib" mediante joblib.dump
RUTA_MODELO = "modelo_modulo5.joblib"
joblib.dump(modelo_optimizado, RUTA_MODELO)
print(f"\nModelo optimizado guardado en: {RUTA_MODELO}")

# 4. Comprueba que el archivo puede cargarse y producir predicciones
modelo_cargado = joblib.load(RUTA_MODELO)

# Realizar una pequeña predicción para verificar
predicciones_prueba_cargado = modelo_cargado.predict(X_test_cls.head(5))
print(f"\nPredicciones de prueba con el modelo cargado: {predicciones_prueba_cargado}")
print("Verificación de carga exitosa.")

,caracteristica,importancia
7,FS1_mean,0.430599
6,EPS1_mean,0.145367
2,PS3_mean,0.102343
0,PS1_mean,0.075378
10,TS2_mean,0.045483
1,PS2_mean,0.032589
9,TS1_mean,0.029998
12,TS4_mean,0.026450
5,PS6_mean,0.024234
11,TS3_mean,0.022473



Modelo optimizado guardado en: modelo_modulo5.joblib

Predicciones de prueba con el modelo cargado: [0. 2. 1. 1. 1.]
Verificación de carga exitosa.


## 8. Conclusión técnica - 1 punto

Crea el diccionario `conclusion_tecnica` con textos propios y estas claves:

- `modelo_clasificacion`
- `metrica_prioritaria`
- `error_mas_costoso`
- `modelo_regresion`
- `interpretacion_mae`
- `limitacion`
- `accion_recomendada`

La conclusión debe relacionar desempeño, tipo de error, tolerancia y decisión de mantenimiento.

In [29]:
# RESPUESTA DEL ESTUDIANTE - TAREA 7
# Escribe tu solución debajo.



In [30]:
conclusion_tecnica = {
    "modelo_clasificacion": "El modelo de clasificación seleccionado fue Random Forest, destacando por su alta capacidad de identificar fugas en la bomba (f1_macro de ~0.99).",
    "metrica_prioritaria": "Para la clasificación, la métrica prioritaria fue f1_macro. Esta métrica es crucial en contextos donde tanto los falsos positivos como los falsos negativos tienen costes significativos, ofreciendo un buen equilibrio entre precisión y recall.",
    "error_mas_costoso": "En el mantenimiento predictivo, un Falso Negativo (no detectar una fuga de bomba cuando sí existe) es el error más costoso. Puede llevar a paradas imprevistas, daños mayores al equipo y altos costos de reparación.",
    "modelo_regresion": "El modelo de regresión seleccionado fue Random Forest Regresión. Sin embargo, su R2 negativo indica que no es un buen predictor de la eficiencia de enfriamiento, rindiendo peor que simplemente predecir la media.",
    "interpretacion_mae": "El MAE del modelo de regresión fue aproximadamente 0.49. Esto significa que, en promedio, la predicción de la eficiencia de enfriamiento se desvía en 0.49 unidades del valor real. Aunque el valor absoluto es pequeño, el bajo R2 sugiere que el modelo tiene dificultades para capturar la variabilidad.",
    "limitacion": "La principal limitación identificada es el rendimiento insatisfactorio del modelo de regresión. El bajo R2 sugiere que las características disponibles pueden no ser lo suficientemente informativas o que el fenómeno es inherentemente más complejo y requiere más datos o una ingeniería de características avanzada.",
    "accion_recomendada": "Se recomienda un monitoreo continuo del modelo de clasificación por su alto rendimiento y potencial de despliegue. Para la regresión, se sugiere una investigación más profunda de nuevas características o el uso de modelos más complejos, dada la actual ineficacia para predecir la eficiencia de enfriamiento."
}

# Mostrar la conclusión técnica para verificación
print("Conclusión Técnica:")
for key, value in conclusion_tecnica.items():
    print(f"- {key.replace('_', ' ').title()}: {value}")

Conclusión Técnica:
- Modelo Clasificacion: El modelo de clasificación seleccionado fue Random Forest, destacando por su alta capacidad de identificar fugas en la bomba (f1_macro de ~0.99).
- Metrica Prioritaria: Para la clasificación, la métrica prioritaria fue f1_macro. Esta métrica es crucial en contextos donde tanto los falsos positivos como los falsos negativos tienen costes significativos, ofreciendo un buen equilibrio entre precisión y recall.
- Error Mas Costoso: En el mantenimiento predictivo, un Falso Negativo (no detectar una fuga de bomba cuando sí existe) es el error más costoso. Puede llevar a paradas imprevistas, daños mayores al equipo y altos costos de reparación.
- Modelo Regresion: El modelo de regresión seleccionado fue Random Forest Regresión. Sin embargo, su R2 negativo indica que no es un buen predictor de la eficiencia de enfriamiento, rindiendo peor que simplemente predecir la media.
- Interpretacion Mae: El MAE del modelo de regresión fue aproximadamente 0.49.

## 9. Calificador automático

Ejecuta esta celda después de completar todas las tareas. Si realizas una corrección, vuelve a ejecutar la tarea modificada y después el calificador.

**No modifiques el calificador.**

In [31]:
# CALIFICADOR AUTOMÁTICO - NO MODIFICAR
from pathlib import Path as _Path
from sklearn.model_selection import GridSearchCV as _GridSearchCV
from sklearn.dummy import DummyClassifier as _DummyClassifier, DummyRegressor as _DummyRegressor

def _obtener(nombre, valor_por_defecto=None):
    return globals().get(nombre, valor_por_defecto)

def _es_modelo_ajustado(modelo):
    if modelo is None or not hasattr(modelo, "predict"):
        return False
    return any(nombre.endswith("_") for nombre in vars(modelo)) or hasattr(modelo, "steps")

detalle = []

def _registrar(criterio, puntos, maximo, evidencia):
    detalle.append({
        "criterio": criterio,
        "puntos": round(float(puntos), 2),
        "maximo": float(maximo),
        "evidencia": evidencia,
    })

# 1. Formulación - 2 puntos
p = 0
X_ = _obtener("X")
yc_ = _obtener("y_clasificacion")
yr_ = _obtener("y_regresion")
rev_ = _obtener("revision_fuga", {})
if isinstance(X_, pd.DataFrame) and list(X_.columns) == FEATURE_COLUMNS and len(X_) == len(datos):
    p += 0.75
if isinstance(yc_, pd.Series) and yc_.name == TARGET_CLASSIFICATION and len(yc_) == len(datos):
    p += 0.35
if isinstance(yr_, pd.Series) and yr_.name == TARGET_REGRESSION and len(yr_) == len(datos):
    p += 0.35
if isinstance(rev_, dict) and rev_.get("hay_fuga") is False and len(rev_.get("columnas_objetivo_en_X", [])) == 0:
    p += 0.55
_registrar("Formulación y control de fuga", p, 2, "X, objetivos y revisión de fuga")

# 2. Particiones - 2 puntos
p = 0
part_cls = [_obtener(n) for n in ["X_train_cls", "X_test_cls", "y_train_cls", "y_test_cls"]]
if all(v is not None for v in part_cls):
    Xtr, Xte, ytr, yte = part_cls
    if len(Xtr) + len(Xte) == len(datos) and set(Xtr.index).isdisjoint(set(Xte.index)):
        p += 0.5
    if 0.23 <= len(Xte) / len(datos) <= 0.27 and set(ytr.unique()) == set(yc_.unique()) and set(yte.unique()) == set(yc_.unique()):
        p += 0.5
part_reg = [_obtener(n) for n in ["X_train_reg", "X_test_reg", "y_train_reg", "y_test_reg"]]
if all(v is not None for v in part_reg):
    Xtr, Xte, ytr, yte = part_reg
    if len(Xtr) + len(Xte) == len(datos) and 0.23 <= len(Xte) / len(datos) <= 0.27:
        p += 0.5
    if len(Xtr) and len(Xte) and max(Xtr.index) < min(Xte.index) and list(Xtr.index) == sorted(Xtr.index) and list(Xte.index) == sorted(Xte.index):
        p += 0.5
_registrar("Particiones reproducibles", p, 2, "Estratificación y reserva cronológica")

# 3. Clasificación - 5 puntos
p = 0
base_cls = _obtener("modelo_base_cls")
mods_cls = _obtener("modelos_clasificacion", {})
preds_cls = _obtener("predicciones_clasificacion", {})
res_cls = _obtener("resultados_clasificacion")
sel_cls = _obtener("modelo_clasificacion_seleccionado")
mc = _obtener("matriz_confusion")
if isinstance(base_cls, _DummyClassifier) and hasattr(base_cls, "classes_"):
    p += 0.75
if isinstance(mods_cls, dict) and len(mods_cls) >= 3 and all(_es_modelo_ajustado(m) for m in mods_cls.values()):
    nombres = {m.__class__.__name__ for m in mods_cls.values()}
    if any("Pipeline" in n for n in nombres) or any(hasattr(m, "steps") for m in mods_cls.values()):
        p += 0.5
    if any("DecisionTreeClassifier" == n for n in nombres) and any("RandomForestClassifier" == n for n in nombres):
        p += 0.75
if isinstance(preds_cls, dict) and len(preds_cls) >= 3 and all(len(v) == len(_obtener("y_test_cls", [])) for v in preds_cls.values()):
    p += 0.5
cols_cls = {"modelo", "accuracy", "precision_macro", "recall_macro", "f1_macro"}
if isinstance(res_cls, pd.DataFrame) and cols_cls.issubset(res_cls.columns) and len(res_cls) >= 4:
    metricas = res_cls[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].apply(pd.to_numeric, errors="coerce")
    if metricas.notna().all().all() and ((metricas >= 0) & (metricas <= 1)).all().all():
        p += 1.25
if _es_modelo_ajustado(sel_cls):
    p += 0.5
if isinstance(mc, np.ndarray) and mc.shape == (len(np.unique(_obtener("y_test_cls", []))),) * 2 and mc.sum() == len(_obtener("y_test_cls", [])):
    p += 0.75
_registrar("Clasificación y tipos de error", p, 5, "Base, tres modelos, métricas y matriz")

# 4. Regresión - 4 puntos
p = 0
base_reg = _obtener("modelo_base_reg")
mods_reg = _obtener("modelos_regresion", {})
preds_reg = _obtener("predicciones_regresion", {})
res_reg = _obtener("resultados_regresion")
sel_reg = _obtener("modelo_regresion_seleccionado")
if isinstance(base_reg, _DummyRegressor) and hasattr(base_reg, "constant_"):
    p += 0.6
if isinstance(mods_reg, dict) and len(mods_reg) >= 3 and all(_es_modelo_ajustado(m) for m in mods_reg.values()):
    nombres = {m.__class__.__name__ for m in mods_reg.values()}
    if {"LinearRegression", "DecisionTreeRegressor", "RandomForestRegressor"}.issubset(nombres):
        p += 1.0
if isinstance(preds_reg, dict) and len(preds_reg) >= 3 and all(len(v) == len(_obtener("y_test_reg", [])) for v in preds_reg.values()):
    p += 0.4
cols_reg = {"modelo", "MAE", "RMSE", "R2"}
if isinstance(res_reg, pd.DataFrame) and cols_reg.issubset(res_reg.columns) and len(res_reg) >= 4:
    metricas = res_reg[["MAE", "RMSE", "R2"]].apply(pd.to_numeric, errors="coerce")
    if metricas.notna().all().all() and np.isfinite(metricas.to_numpy()).all() and (metricas[["MAE", "RMSE"]] >= 0).all().all():
        p += 1.5
if _es_modelo_ajustado(sel_reg):
    p += 0.5
_registrar("Regresión y magnitud del error", p, 4, "Base, tres regresores y métricas")

# 5. Optimización - 4 puntos
p = 0
pipe = _obtener("pipeline_busqueda")
grid = _obtener("param_grid", {})
cv = _obtener("cv_estratificada")
busq = _obtener("busqueda")
opt = _obtener("modelo_optimizado")
met_opt = _obtener("metricas_optimizadas", {})
if hasattr(pipe, "steps") and any(nombre == "imputer" for nombre, _ in pipe.steps) and any(nombre == "model" for nombre, _ in pipe.steps):
    p += 0.75
if isinstance(grid, dict) and all(any(k.endswith(sufijo) for k in grid) for sufijo in ["n_estimators", "max_depth", "min_samples_leaf"]) and all(len(v) >= 2 for v in grid.values()):
    p += 0.75
if cv is not None and getattr(cv, "n_splits", 0) >= 3:
    p += 0.5
if isinstance(busq, _GridSearchCV) and hasattr(busq, "best_estimator_") and getattr(busq, "scoring", None) == "f1_macro":
    p += 1.25
if opt is not None and opt is getattr(busq, "best_estimator_", None):
    p += 0.25
claves_opt = {"accuracy", "precision_macro", "recall_macro", "f1_macro"}
if isinstance(met_opt, dict) and claves_opt.issubset(met_opt) and all(np.isfinite(float(met_opt[k])) and 0 <= float(met_opt[k]) <= 1 for k in claves_opt):
    p += 0.5
_registrar("Optimización sin abrir la prueba", p, 4, "Pipeline, CV, GridSearch y prueba final")

# 6. Interpretación y persistencia - 2 puntos
p = 0
imp = _obtener("importancias")
ruta = _obtener("RUTA_MODELO")
if isinstance(imp, pd.DataFrame) and {"caracteristica", "importancia"}.issubset(imp.columns) and len(imp) >= 10:
    vals = pd.to_numeric(imp["importancia"], errors="coerce")
    if vals.notna().all() and (vals >= 0).all() and vals.is_monotonic_decreasing:
        p += 1.0
if isinstance(ruta, str) and _Path(ruta).exists():
    try:
        cargado = joblib.load(ruta)
        pred = cargado.predict(_obtener("X_test_cls").iloc[:3])
        if len(pred) == 3:
            p += 1.0
    except Exception:
        pass
_registrar("Interpretación y persistencia", p, 2, "Importancias y pipeline guardado")

# 7. Conclusión - 1 punto
p = 0
conclusion = _obtener("conclusion_tecnica", {})
claves_conclusion = {
    "modelo_clasificacion", "metrica_prioritaria", "error_mas_costoso",
    "modelo_regresion", "interpretacion_mae", "limitacion", "accion_recomendada",
}
if isinstance(conclusion, dict) and claves_conclusion.issubset(conclusion):
    textos = [str(conclusion[k]).strip() for k in claves_conclusion]
    if all(len(t) >= 12 for t in textos):
        p = 1.0
_registrar("Conclusión técnica", p, 1, "Desempeño, riesgo, límite y acción")

tabla_calificacion = pd.DataFrame(detalle)
PUNTAJE_TOTAL = round(tabla_calificacion["puntos"].sum(), 2)
tabla_calificacion.loc[len(tabla_calificacion)] = ["TOTAL", PUNTAJE_TOTAL, 20.0, ""]

display(tabla_calificacion)
print(f"PUNTAJE AUTOMÁTICO: {PUNTAJE_TOTAL:.2f} / 20.00")
if not str(_obtener("NOMBRE_COMPLETO", "")).strip():
    print("ADVERTENCIA: completa NOMBRE_COMPLETO antes de entregar.")


,criterio,puntos,maximo,evidencia
0,Formulación y control de fuga,2.0,2.0,"X, objetivos y revisión de fuga"
1,Particiones reproducibles,2.0,2.0,Estratificación y reserva cronológica
2,Clasificación y tipos de error,5.0,5.0,"Base, tres modelos, métricas y matriz"
3,Regresión y magnitud del error,4.0,4.0,"Base, tres regresores y métricas"
4,Optimización sin abrir la prueba,4.0,4.0,"Pipeline, CV, GridSearch y prueba final"
5,Interpretación y persistencia,2.0,2.0,Importancias y pipeline guardado
6,Conclusión técnica,1.0,1.0,"Desempeño, riesgo, límite y acción"
7,TOTAL,20.0,20.0,


PUNTAJE AUTOMÁTICO: 20.00 / 20.00


## 10. Entrega

Antes de descargar el notebook:

1. Ejecuta todas las celdas en orden.
2. Verifica que las tablas de resultados y la calificación sean visibles.
3. Guarda el archivo.
4. Descarga una copia `.ipynb`.
5. Renómbrala como `Apellido_Nombre_Evaluacion_Modulo5.ipynb`.
6. Entrega el notebook completo, no solamente capturas de pantalla.

### Fuente del conjunto de datos

Helwig, N., Pignanelli, E. y Schütze, A. (2015). *Condition Monitoring of Hydraulic Systems*. UCI Machine Learning Repository. https://doi.org/10.24432/C5CW21
